{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "f5948e39",
   "metadata": {},
   "source": [
    "# 🌩️ ISRO Hackathon Challenge No. 9\n",
    "## 🛰️ TCC DETECTION: Tropical Cloud Clusters Detection Using INSAT Satellite Data\n",
    "\n",
    "### 🧾 Project Description:\n",
    "This project aims to develop an AI/ML-based algorithm that automatically identifies Tropical Cloud Clusters (TCCs) using half-hourly Infrared Brightness Temperature (IRBT) satellite data from INSAT-3D. These clusters play a crucial role in tropical weather systems and cyclogenesis (formation of cyclones). The model will detect the presence of TCCs and extract all ISRO-specified features like size, pixel count, radii, temperature statistics, and optionally cloud-top height — potentially contributing to real-time weather forecasting systems.\n",
    "Since actual data is unavailable, a curated dataset of 35+ synthetic/sampled IR-like images is used to mimic the real scenario.\n",
    "In the extended version of this project, the detected cloud clusters are also classified into severity levels such as Normal, Rain, Heavy Rain, Storm, or Cyclone based on their size and brightness temperature characteristics.\n",
    "### 🔑 Key Features Extracted per Image\n",
    "| Parameter               | Meaning                              |\n",
    "| ----------------------- | ------------------------------------ |\n",
    "| 📍 Convective Center    | Latitude, Longitude of coldest pixel |\n",
    "| 🔢 Pixel Count          | No. of pixels in the TCC (size)      |\n",
    "| 🌡️ Mean Tb             | Average Brightness Temp (°C)         |\n",
    "| 🌡️ Min Tb              | Coldest pixel temp (°C)              |\n",
    "| 🌡️ Median Tb           | Median Brightness Temp               |\n",
    "| 📏 Radius (Max/Min/Avg) | Dist. from center to edge            |\n",
    "| 🏔️ Cloud Top Height    | Estimated from IR brightness temp    |\n",
    "\n",
    "## 🎯 Project Objectives:\n",
    "- Automatically detect Tropical Cloud Clusters from satellite data.\n",
    "- Extract key parameters: center, min/max/mean Tb, radii, size, pixel count.\n",
    "- Provide an interactive interface for demo/validation.\n",
    "- Build a system suitable for ISRO-level use cases and research.\n",
    "\n",
    "## Benefits:\n",
    "- Supports cyclone early warning systems\n",
    "- Useful for weather research and forecasting\n",
    "- Scalable to large regions and multiple time zones\n",
    "\n",
    "## ⚠️Challenges:\n",
    "- Limited access to high-quality labeled satellite data\n",
    "- Variability in IRBRT due to geography and cloud height\n",
    "- Real-time accuracy demands speed and efficiency\n",
    "- Preprocessing large satellite datasets in HDF/NetCDF format.\n",
    "-Precise threshold tuning for Tb (e.g., 240K standard for convective clouds).\n",
    "- Handling false positives/negatives in cloud detection.\n",
    "- Lack of directly labeled TCC data for supervised learning.\n",
    "\n",
    "## 🌍Scope:\n",
    "- Works on IR images from INSAT-3D (can adapt to MODIS, etc.)\n",
    "- Apply thresholding + clustering to extract TCCs\n",
    "- Usable for tracking cloud clusters across time for early cyclone alerts.\n",
    "- Optional: CNN model for enhancement\n",
    "- Expandable to estimate rainfall or storm strength by integrating with other bands (WV, MIR).\n",
    "- Deploy model using Gradio\n",
    "\n",
    "## 🧰 Core Libraries & Tools:\n",
    "| Library                 | Purpose                                |\n",
    "| ----------------------- | -------------------------------------- |\n",
    "| `numpy`, `pandas`       | Data manipulation                      |\n",
    "| `matplotlib`, `seaborn` | Visualizations                         |\n",
    "| `tensorflow`, `keras`   | Deep learning model                    |\n",
    "| `sklearn`               | Metrics & preprocessing                |\n",
    "| `scipy.ndimage`         | Connected component labeling, distance |\n",
    "| `gradio`                | UI deployment                          |\n",
    "| `netCDF4`, `pyhdf`      | Satellite data reading                 |\n",
    "\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "5d3946e5",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-16T12:12:58.463364Z",
     "iopub.status.busy": "2026-08-16T12:12:58.463364Z",
     "iopub.status.idle": "2026-08-16T12:13:06.064893Z",
     "shell.execute_reply": "2026-08-16T12:13:06.063884Z"
    }
   },
   "outputs": [
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "C:\\Users\\janak\\OneDrive\\Desktop\\TROPICAL-CLOUD-CLUSTER-TCC-DETECTION-main\\.venv312\\Lib\\site-packages\\tqdm\\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html\n",
      "  from .autonotebook import tqdm as notebook_tqdm\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "✅ All libraries loaded successfully\n"
     ]
    }
   ],
   "source": [
    "import os\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import cv2\n",
    "import pandas as pd\n",
    "import seaborn as sns\n",
    "from scipy.ndimage import label, center_of_mass\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.metrics import classification_report, confusion_matrix\n",
    "from tensorflow.keras import Model\n",
    "from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout\n",
    "from tensorflow.keras.callbacks import EarlyStopping\n",
    "from tensorflow.keras.preprocessing.image import ImageDataGenerator\n",
    "import gradio as gr\n",
    "import random\n",
    "\n",
    "print(\"✅ All libraries loaded successfully\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6dce3549",
   "metadata": {},
   "source": [
    "### 📊 STEP-BY-STEP IMPLEMENTATION\n",
    "# 🔍 Step 1: Explore & Preprocess Data\n",
    "\n",
    "We start by loading and visualizing simulated IR images, then apply preprocessing steps to:\n",
    "- Convert grayscale images into numerical arrays\n",
    "- Apply a temperature threshold (e.g., Tb < 220 K) to identify cold cloud clusters\n",
    "- Extract connected components\n",
    "- Calculate cluster-level metrics\n",
    "\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "2225ff00",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-16T12:13:06.067512Z",
     "iopub.status.busy": "2026-08-16T12:13:06.066399Z",
     "iopub.status.idle": "2026-08-16T12:13:06.297232Z",
     "shell.execute_reply": "2026-08-16T12:13:06.295218Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "✅ Processed dummy_image_0.png with 436 clusters\n",
      "✅ Processed dummy_image_1.png with 455 clusters\n",
      "✅ Processed dummy_image_2.png with 454 clusters\n",
      "✅ Processed dummy_image_3.png with 431 clusters\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "✅ Processed dummy_image_4.png with 414 clusters\n",
      "📁 Saved extracted features to cluster_features.csv\n"
     ]
    }
   ],
   "source": [
    "# ✅ Required Libraries\n",
    "import os\n",
    "import numpy as np\n",
    "import cv2\n",
    "import pandas as pd\n",
    "from scipy.ndimage import label, center_of_mass\n",
    "\n",
    "# 📁 Folder Paths\n",
    "image_folder = \"tcc_irbrt_images\"            # ✅ Replace with your folder path\n",
    "output_folder = \"tcc_cluster_output\"\n",
    "os.makedirs(output_folder, exist_ok=True)\n",
    "\n",
    "# 📊 Data Collection\n",
    "all_cluster_data = []\n",
    "\n",
    "# 🌀 Process each image\n",
    "for filename in sorted(os.listdir(image_folder)):\n",
    "    if filename.endswith(\".png\") or filename.endswith(\".jpg\"):\n",
    "        img_path = os.path.join(image_folder, filename)\n",
    "        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)\n",
    "        if img is None:\n",
    "            continue\n",
    "\n",
    "        # ⚙️ Apply thresholding to simulate TCC detection (e.g., Tb < 220K ≈ pixel < 100)\n",
    "        binary = np.where(img < 100, 1, 0).astype(np.uint8)\n",
    "        labeled_array, num_features = label(binary)\n",
    "\n",
    "        # 📌 Process each detected cluster\n",
    "        for cluster_id in range(1, num_features + 1):\n",
    "            cluster_mask = labeled_array == cluster_id\n",
    "            coords = np.argwhere(cluster_mask)\n",
    "            if coords.size == 0:\n",
    "                continue\n",
    "\n",
    "            y_coords, x_coords = coords[:, 0], coords[:, 1]\n",
    "            Tb_values = img[y_coords, x_coords].astype(np.float32)  # ⚠️ Convert to float\n",
    "\n",
    "            min_Tb = float(Tb_values.min())\n",
    "            mean_Tb = float(Tb_values.mean())\n",
    "            median_Tb = float(np.median(Tb_values))\n",
    "            center_y, center_x = center_of_mass(cluster_mask)\n",
    "\n",
    "            # 📏 Calculate radius-related metrics\n",
    "            distances = np.sqrt((x_coords - center_x) ** 2 + (y_coords - center_y) ** 2)\n",
    "            min_radius = float(distances.min())\n",
    "            max_radius = float(distances.max())\n",
    "            mean_radius = float(distances.mean())\n",
    "\n",
    "            # 🏔️ Estimate cloud top height safely\n",
    "            cloud_top_height = (290.0 - min_Tb) * 0.2  # 290 is float\n",
    "\n",
    "            all_cluster_data.append({\n",
    "                'Image': filename,\n",
    "                'PixelCount': len(coords),\n",
    "                'MeanTb': round(mean_Tb, 2),\n",
    "                'MinTb': round(min_Tb, 2),\n",
    "                'MedianTb': round(median_Tb, 2),\n",
    "                'CenterX': round(center_x, 2),\n",
    "                'CenterY': round(center_y, 2),\n",
    "                'MinRadius': round(min_radius, 2),\n",
    "                'MaxRadius': round(max_radius, 2),\n",
    "                'MeanRadius': round(mean_radius, 2),\n",
    "                'CloudTopHeight': round(cloud_top_height, 2)\n",
    "            })\n",
    "\n",
    "        print(f\"✅ Processed {filename} with {num_features} clusters\")\n",
    "\n",
    "# 💾 Save features to CSV\n",
    "features_df = pd.DataFrame(all_cluster_data)\n",
    "features_df.to_csv(os.path.join(output_folder, \"cluster_features.csv\"), index=False)\n",
    "print(\"📁 Saved extracted features to cluster_features.csv\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "bcbe3a64",
   "metadata": {},
   "source": [
    "# 🧠 Step 2: Model Selection\n",
    "\n",
    "We will use a basic Convolutional Neural Network (CNN) to classify whether\n",
    "an IR image contains a tropical cloud cluster (TCC) or not.\n",
    "\n",
    "Input: 64x64 grayscale IR image patch  \n",
    "Output: Binary label - TCC (1) or No TCC (0)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "d088f6d7",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-16T12:13:06.300303Z",
     "iopub.status.busy": "2026-08-16T12:13:06.300303Z",
     "iopub.status.idle": "2026-08-16T12:13:06.514876Z",
     "shell.execute_reply": "2026-08-16T12:13:06.513863Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\">Model: \"functional\"</span>\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1mModel: \"functional\"\u001b[0m\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\">┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓\n",
       "┃<span style=\"font-weight: bold\"> Layer (type)                    </span>┃<span style=\"font-weight: bold\"> Output Shape           </span>┃<span style=\"font-weight: bold\">       Param # </span>┃\n",
       "┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩\n",
       "│ input_layer (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">InputLayer</span>)        │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">64</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">64</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">1</span>)      │             <span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ conv2d (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Conv2D</span>)                 │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">62</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">62</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">32</span>)     │           <span style=\"color: #00af00; text-decoration-color: #00af00\">320</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ max_pooling2d (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">MaxPooling2D</span>)    │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">31</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">31</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">32</span>)     │             <span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ conv2d_1 (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Conv2D</span>)               │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">29</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">29</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">64</span>)     │        <span style=\"color: #00af00; text-decoration-color: #00af00\">18,496</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ max_pooling2d_1 (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">MaxPooling2D</span>)  │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">14</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">14</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">64</span>)     │             <span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ flatten (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Flatten</span>)               │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">12544</span>)          │             <span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Dense</span>)                   │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">64</span>)             │       <span style=\"color: #00af00; text-decoration-color: #00af00\">802,880</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dropout (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Dropout</span>)               │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">64</span>)             │             <span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense_1 (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Dense</span>)                 │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">1</span>)              │            <span style=\"color: #00af00; text-decoration-color: #00af00\">65</span> │\n",
       "└─────────────────────────────────┴────────────────────────┴───────────────┘\n",
       "</pre>\n"
      ],
      "text/plain": [
       "┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓\n",
       "┃\u001b[1m \u001b[0m\u001b[1mLayer (type)                   \u001b[0m\u001b[1m \u001b[0m┃\u001b[1m \u001b[0m\u001b[1mOutput Shape          \u001b[0m\u001b[1m \u001b[0m┃\u001b[1m \u001b[0m\u001b[1m      Param #\u001b[0m\u001b[1m \u001b[0m┃\n",
       "┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩\n",
       "│ input_layer (\u001b[38;5;33mInputLayer\u001b[0m)        │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m64\u001b[0m, \u001b[38;5;34m64\u001b[0m, \u001b[38;5;34m1\u001b[0m)      │             \u001b[38;5;34m0\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ conv2d (\u001b[38;5;33mConv2D\u001b[0m)                 │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m62\u001b[0m, \u001b[38;5;34m62\u001b[0m, \u001b[38;5;34m32\u001b[0m)     │           \u001b[38;5;34m320\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ max_pooling2d (\u001b[38;5;33mMaxPooling2D\u001b[0m)    │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m31\u001b[0m, \u001b[38;5;34m31\u001b[0m, \u001b[38;5;34m32\u001b[0m)     │             \u001b[38;5;34m0\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ conv2d_1 (\u001b[38;5;33mConv2D\u001b[0m)               │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m29\u001b[0m, \u001b[38;5;34m29\u001b[0m, \u001b[38;5;34m64\u001b[0m)     │        \u001b[38;5;34m18,496\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ max_pooling2d_1 (\u001b[38;5;33mMaxPooling2D\u001b[0m)  │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m14\u001b[0m, \u001b[38;5;34m14\u001b[0m, \u001b[38;5;34m64\u001b[0m)     │             \u001b[38;5;34m0\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ flatten (\u001b[38;5;33mFlatten\u001b[0m)               │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m12544\u001b[0m)          │             \u001b[38;5;34m0\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense (\u001b[38;5;33mDense\u001b[0m)                   │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m64\u001b[0m)             │       \u001b[38;5;34m802,880\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dropout (\u001b[38;5;33mDropout\u001b[0m)               │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m64\u001b[0m)             │             \u001b[38;5;34m0\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense_1 (\u001b[38;5;33mDense\u001b[0m)                 │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m1\u001b[0m)              │            \u001b[38;5;34m65\u001b[0m │\n",
       "└─────────────────────────────────┴────────────────────────┴───────────────┘\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Total params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">821,761</span> (3.13 MB)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Total params: \u001b[0m\u001b[38;5;34m821,761\u001b[0m (3.13 MB)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Trainable params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">821,761</span> (3.13 MB)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Trainable params: \u001b[0m\u001b[38;5;34m821,761\u001b[0m (3.13 MB)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Non-trainable params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> (0.00 B)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Non-trainable params: \u001b[0m\u001b[38;5;34m0\u001b[0m (0.00 B)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "def build_cnn_model():\n",
    "    inputs = Input(shape=(64, 64, 1))\n",
    "    x = Conv2D(32, (3, 3), activation='relu')(inputs)\n",
    "    x = MaxPooling2D((2, 2))(x)\n",
    "    x = Conv2D(64, (3, 3), activation='relu')(x)\n",
    "    x = MaxPooling2D((2, 2))(x)\n",
    "    x = Flatten()(x)\n",
    "    x = Dense(64, activation='relu')(x)\n",
    "    x = Dropout(0.5)(x)\n",
    "    outputs = Dense(1, activation='sigmoid')(x)\n",
    "\n",
    "    model = Model(inputs, outputs)\n",
    "    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])\n",
    "    return model\n",
    "\n",
    "model = build_cnn_model()\n",
    "model.summary()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4c1b9cb6",
   "metadata": {},
   "source": [
    "# 🏋️ Step 3: Model Training\n",
    "Now we train the CNN model. Dataset must be organized as:\n",
    "\n",
    "./dataset/\n",
    "  └── TCC/     (images labeled 1)\n",
    "  └── NoTCC/   (images labeled 0)\n",
    "\n",
    "We will:\n",
    "- Load and resize all images to 64x64\n",
    "- Normalize pixel values\n",
    "- Encode labels (1 for TCC, 0 for NoTCC)\n",
    "- Split into train and test sets\n",
    "- Train the CNN with EarlyStopping\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "44376a7a",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-16T12:13:06.518874Z",
     "iopub.status.busy": "2026-08-16T12:13:06.517874Z",
     "iopub.status.idle": "2026-08-16T12:13:08.964739Z",
     "shell.execute_reply": "2026-08-16T12:13:08.964739Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Dataset path: ../dataset\n",
      "Exists: True\n",
      "Found 8 images belonging to 2 classes.\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Found 2 images belonging to 2 classes.\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Epoch 1/20\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 1s/step - accuracy: 0.6250 - loss: 0.6746"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m2s\u001b[0m 2s/step - accuracy: 0.6250 - loss: 0.6746 - val_accuracy: 0.5000 - val_loss: 0.7047\n"
     ]
    },
@@ -478,15 +478,15 @@
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 59ms/step - accuracy: 0.5000 - loss: 0.7919"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 125ms/step - accuracy: 0.5000 - loss: 0.7919 - val_accuracy: 0.5000 - val_loss: 0.6915\n"
     ]
    },
@@ -501,15 +501,15 @@
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 50ms/step - accuracy: 0.3750 - loss: 0.8451"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 113ms/step - accuracy: 0.3750 - loss: 0.8451 - val_accuracy: 0.5000 - val_loss: 0.6986\n"
     ]
    },
@@ -524,15 +524,15 @@
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 45ms/step - accuracy: 0.6250 - loss: 0.6573"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 92ms/step - accuracy: 0.6250 - loss: 0.6573 - val_accuracy: 0.5000 - val_loss: 0.6957\n"
     ]
    },
@@ -547,15 +547,15 @@
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 53ms/step - accuracy: 0.6250 - loss: 0.7509"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 115ms/step - accuracy: 0.6250 - loss: 0.7509 - val_accuracy: 0.5000 - val_loss: 0.6964\n"
     ]
    },
@@ -570,15 +570,15 @@
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 68ms/step - accuracy: 0.6250 - loss: 0.6479"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 121ms/step - accuracy: 0.6250 - loss: 0.6479 - val_accuracy: 0.5000 - val_loss: 0.6955\n"
     ]
    },
@@ -593,15 +593,15 @@
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 49ms/step - accuracy: 0.7500 - loss: 0.6505"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 105ms/step - accuracy: 0.7500 - loss: 0.6505 - val_accuracy: 0.5000 - val_loss: 0.6984\n"
     ]
    },
@@ -766,15 +766,15 @@
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 99ms/step"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 108ms/step\n"
     ]
    },
@@ -855,127 +855,127 @@
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 80ms/step"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 92ms/step\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 24ms/step"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 50ms/step\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 16ms/step"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 36ms/step\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 34ms/step"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 80ms/step\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 38ms/step"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 69ms/step\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 30ms/step"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 42ms/step\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 18ms/step"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 31ms/step\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\r",
      "\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 22ms/step"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r",
      "\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\b\r\n",
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 48ms/step\n"
     ]
    },